In [13]:
import ArcFace
import tensorflow as tf

In [2]:
base_model = ArcFace.load_model()

2025-07-16 09:10:17.890256: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


In [4]:
base_model.output_shape

(None, 512)

In [11]:
# ArcFace expects 112x112 pixel inputs
IMG_SIZE = (112, 112)
BATCH_SIZE = 32
import os
BASE_DIR = os.getcwd()
DATASET_PATH = os.path.join(BASE_DIR, 'data', 'images')

In [14]:
# Load training data
train_dataset = tf.keras.utils.image_dataset_from_directory(
    DATASET_PATH,
    validation_split=0.2,
    subset="training",
    seed=123,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
)

# Load validation data
validation_dataset = tf.keras.utils.image_dataset_from_directory(
    DATASET_PATH,
    validation_split=0.2,
    subset="validation",
    seed=123,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
)

# Get the class names (identities)
class_names = train_dataset.class_names
num_classes = len(class_names)
print(f"Found {num_classes} classes: {class_names}")

# Preprocessing function
# ArcFace models are typically trained on images normalized between -1 and 1
def preprocess_image(image, label):
    image = tf.cast(image, tf.float32)
    image = (image / 127.5) - 1  # Normalize to the range [-1, 1]
    return image, label

train_dataset = train_dataset.map(preprocess_image).cache().prefetch(buffer_size=tf.data.AUTOTUNE)
validation_dataset = validation_dataset.map(preprocess_image).cache().prefetch(buffer_size=tf.data.AUTOTUNE)

Found 8792 files belonging to 235 classes.
Using 7034 files for training.
Found 8792 files belonging to 235 classes.
Using 1758 files for validation.
Found 235 classes: ['Ca Sĩ Lamoon Diễm Hằng', 'ca sĩ Akira Phan', 'ca sĩ Anh Thơ', 'ca sĩ Bằng Kiều', 'ca sĩ Bảo Anh', 'ca sĩ Bảo Thy', 'ca sĩ Bigdaddy', 'ca sĩ Bích Phương', 'ca sĩ Bùi Anh Tuấn', 'ca sĩ Cao Thái Sơn', 'ca sĩ Cát Tường', 'ca sĩ Cẩm Ly', 'ca sĩ Chi Dân', 'ca sĩ Dịch Dương Thiên Tỉ', 'ca sĩ Doãn Hiếu', 'ca sĩ Don Nguyễn', 'ca sĩ Dương Triệu Vũ', 'ca sĩ Emily', 'ca sĩ Enrique Iglesias', 'ca sĩ Erik', 'ca sĩ Fanny Trần', 'ca sĩ Gin Tuấn Kiệt', 'ca sĩ Hari Won', 'ca sĩ Hà Anh Tuấn', 'ca sĩ Hà Thanh Xuân', 'ca sĩ Hải Băng', 'ca sĩ Hiền Thục', 'ca sĩ Hoài Lâm', 'ca sĩ Hoàng Thùy Linh', 'ca sĩ Hoàng Yến Chibi', 'ca sĩ Hòa Minzy', 'ca sĩ Hồ Ngọc Hà', 'ca sĩ Hồ Quang Hiếu', 'ca sĩ Hồ Quỳnh Hương', 'ca sĩ Hồ Văn Cường', 

## New model for fine-tuning

In [17]:
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.models import Model

# First, freeze the base model layers so they are not updated during training
base_model.trainable = False

# Create the new layers
inputs = base_model.input
x = Dropout(0.5)(base_model.input) # Apply dropout to the input for regularization
x = base_model(x)  # Pass through the base model to get the 512-dim embedding vector
outputs = Dense(num_classes, activation="softmax")(x) # New output layer for our classes

# Create the final model
model = Model(inputs=inputs, outputs=outputs)

In [31]:
from tensorflow.keras import backend as K

trainable_params = sum([K.count_params(w) for w in model.trainable_weights])
non_trainable_params = sum([K.count_params(w) for w in model.non_trainable_weights])
total_params = trainable_params + non_trainable_params
num_layers = len(model.layers)

def size_in_kb_mb(params):
    kb = params * 4 / 1024          # assuming 4 bytes per param (float32)
    mb = kb / 1024
    return kb, mb

trainable_kb, trainable_mb = size_in_kb_mb(trainable_params)
non_trainable_kb, non_trainable_mb = size_in_kb_mb(non_trainable_params)
total_kb, total_mb = size_in_kb_mb(total_params)

print(f"Number of layers             : {num_layers}")
print(f"Total params                 : {total_params:,}")
print(f"  - Trainable params         : {trainable_params:,}")
print(f"  - Non-trainable params     : {non_trainable_params:,}")
print(f"Memory footprint:")
print(f"  - Trainable: {trainable_kb:.2f} KB ({trainable_mb:.2f} MB)")
print(f"  - Non-trainable: {non_trainable_kb:.2f} KB ({non_trainable_mb:.2f} MB)")
print(f"  - Total: {total_kb:.2f} KB ({total_mb:.2f} MB)")

Number of layers             : 164
Total params                 : 34,285,739
  - Trainable params         : 120,555
  - Non-trainable params     : 34,165,184
Memory footprint:
  - Trainable: 470.92 KB (0.46 MB)
  - Non-trainable: 133457.75 KB (130.33 MB)
  - Total: 133928.67 KB (130.79 MB)


## Train

In [32]:
# Use a low learning rate to avoid damaging the pre-trained weights
optimizer = tf.keras.optimizers.Adam(learning_rate=0.0001)

model.compile(
    optimizer=optimizer,
    loss=tf.keras.losses.SparseCategoricalCrossentropy(), # Use Sparse because labels are integers
    metrics=['accuracy']
)

# Train the model
epochs = 20 # Start with a moderate number of epochs
history = model.fit(
    train_dataset,
    validation_data=validation_dataset,
    epochs=epochs
)

Epoch 1/20
220/220 ━━━━━━━━━━━━━━━━━━━━ 1353s 6s/step - accuracy: 0.0051 - loss: 5.4998 - val_accuracy: 0.0131 - val_loss: 5.3712
Epoch 2/20
 50/220 ━━━━━━━━━━━━━━━━━━━━ 17:24 6s/step - accuracy: 0.0071 - loss: 5.3874

KeyboardInterrupt: 